In [ ]:
# Install necessary libraries
!pip install tensorflow keras numpy opencv-python matplotlib

# Import required libraries
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import img_to_array, load_img  # Import from tensorflow.keras
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths to training and test datasets
train_path = '/content/drive/MyDrive/Test Data'  # Update if your path is different
test_path = '/content/drive/MyDrive/Test Data'  # Update if your path is different

# Set batch size
BatchSize = 64

# Load and display a sample image to verify path
img = load_img(train_path + '/Apple Braeburn 1/321_100.jpg')  # Adjusted path to image
plt.imshow(img)
plt.show()

# Convert the image to an array and display shape
imgA = img_to_array(img)
print(f"Image Shape: {imgA.shape}")

# Create ImageDataGenerators for training and testing
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize the images
    shear_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.3
)

test_datagen = ImageDataGenerator(rescale=1./255)  # Normalize test images

# Generate data from the directories
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(100, 100),
    batch_size=BatchSize,
    color_mode="rgb",  # Corrected spelling
    class_mode="categorical",
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(100, 100),
    batch_size=BatchSize,
    color_mode="rgb",  # Corrected spelling
    class_mode="categorical"
)

# Calculate the number of classes dynamically
num_classes = train_generator.num_classes
print(f"Number of Classes Detected: {num_classes}")

# Create the Convolutional Neural Network (CNN) model
model = Sequential()
model.add(Conv2D(filters=128, kernel_size=3, activation="relu", input_shape=(100, 100, 3)))
model.add(MaxPooling2D())
model.add(Conv2D(filters=64, kernel_size=3, activation="relu"))
model.add(Conv2D(filters=64, kernel_size=3, activation="relu"))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(5000, activation="relu"))
model.add(Dense(1000, activation="relu"))
model.add(Dense(num_classes, activation="softmax"))  # Update number of classes dynamically

# Display the model summary
print(model.summary())

# Compile the model
model.compile(loss="categorical_crossentropy", optimizer="SGD", metrics=["accuracy"])

# Calculate steps per epoch and validation steps (Convert to int)
stepsPerEpoch = int(np.ceil(train_generator.samples / BatchSize))
ValidationSteps = int(np.ceil(test_generator.samples / BatchSize))

# Early stopping to avoid overfitting
stop_early = EarlyStopping(monitor="val_accuracy", patience=5)

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=stepsPerEpoch,  # This should now be an integer
    epochs=50,
    validation_data=test_generator,
    validation_steps=ValidationSteps,  # This should now be an integer
    callbacks=[stop_early]
)

# Save the trained model
model.save('fruit_model.h5')


Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Test Data/Apple Braeburn 1/321_100.jpg'